In [1]:
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm

import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


In [2]:
with open('../../transformed_event_logs/PCR_start_end_train.pickle', 'rb') as f:
    train_data = pickle.load(f)

In [3]:
train_data

,case:concept:name,id:id_start,cpee:activity_start,cpee:instance_start,lifecycle:transition_start,cpee:lifecycle:transition_start,cpee:state_start,time:timestamp_start,data_start,cpee:description_start,...,intercase_n_3__timeout_Match patient data_Wait for plate validation,intercase_n_3__timeout_Receive sample state_Callback timeout,intercase_n_3__timeout_Receive sample state_Export result,intercase_n_3__timeout_Receive sample state_Export to EMS,intercase_n_3__timeout_Receive sample state_Send notification,intercase_n_3__timeout_Send notification_Receive sample state,intercase_n_3__timeout_Wait for plate validation,intercase_n_3__timeout_Wait for plate validation_Match patient data,intercase_n_3__timeout_Wait for plate validation_Receive sample state,intercase_n_3__timeout_Wait for plate validation_Send notification
0,16495,a3,a3,6bf7c694-35d2-4134-a35d-f52eaa51d658,start,activity/calling,0.0,2023-05-20 14:43:28.526,"{'value': None, 'children': [('', {'value': ''...",0.0,...,2,10,0,0,0,0,1,1,0,0
1,14812,a4,a4,786569b6-6fd4-4fe2-a5f4-20c4f39f3307,start,activity/calling,0.0,2023-04-28 13:55:37.424,"{'value': None, 'children': [('', {'value': ''...",0.0,...,3,0,0,0,0,0,0,8,0,0
2,13597,a2,a2,72de7428-a85d-4ee2-9c71-556eaa2bb0b5,start,activity/calling,0.0,2023-04-20 17:12:02.554,"{'value': None, 'children': [('', {'value': ''...",0.0,...,13,0,0,0,0,0,0,11,0,0
3,14616,a4,a4,6d1b2d51-5ae1-46bd-bdae-eb5e04227a28,start,activity/calling,0.0,2023-04-27 13:53:05.267,"{'value': None, 'children': [('', {'value': ''...",0.0,...,17,0,0,0,0,0,0,15,0,0
4,10320,a2,a2,5778bdc7-3d1f-443c-bb24-8dd8e6728b3d,start,activity/calling,0.0,2023-04-03 17:58:20.376,"{'value': None, 'children': [('', {'value': ''...",0.0,...,48,0,0,0,0,0,0,29,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32353,10700,a1,a1,2300a50a-1212-4f41-b8e1-2460f8917827,start,activity/calling,0.0,2023-04-05 15:25:09.959,"{'value': None, 'children': [('', {'value': ''...",0.0,...,33,0,0,0,0,0,0,29,0,0
32355,11283,a4,a4,1c84f4b5-74a3-44b9-b268-01f0ef120814,start,activity/calling,0.0,2023-04-09 14:21:09.416,"{'value': None, 'children': [('', {'value': ''...",0.0,...,2,0,0,0,0,0,0,3,0,0
32356,702,a3,a3,c0ffdfc5-7dd4-4e32-a854-9e5b3469084c,start,activity/calling,0.0,2023-04-11 22:19:51.629,"{'value': None, 'children': [('', {'value': ''...",0.0,...,19,0,0,0,0,0,0,16,0,0
32358,17281,a4,a4,e8018ca3-b31c-4cc7-be10-8974d71275be,start,activity/calling,0.0,2023-06-02 17:00:11.948,"{'value': None, 'children': [('', {'value': ''...",0.0,...,12,0,0,0,0,0,0,11,0,0


In [4]:
list(train_data.columns)

['case:concept:name',
 'id:id_start',
 'cpee:activity_start',
 'cpee:instance_start',
 'lifecycle:transition_start',
 'cpee:lifecycle:transition_start',
 'cpee:state_start',
 'time:timestamp_start',
 'data_start',
 'cpee:description_start',
 'concept:name',
 'concept:endpoint_start',
 'cpee:activity_uuid_start',
 'raw_start',
 'start_timestamp_start',
 'id:id_complete',
 'cpee:activity_complete',
 'cpee:instance_complete',
 'lifecycle:transition_complete',
 'cpee:lifecycle:transition_complete',
 'cpee:state_complete',
 'time:timestamp_complete',
 'data_complete',
 'cpee:description_complete',
 'concept:endpoint_complete',
 'cpee:activity_uuid_complete',
 'raw_complete',
 'start_timestamp_complete',
 'duration',
 'duration_seconds',
 'duration_ms',
 'duration_hours',
 'seconds_in_day',
 'day_of_week',
 'Callback timeout',
 'Export result',
 'Export to EMS',
 'Match patient data',
 'Receive sample state',
 'Send notification',
 'Wait for plate validation',
 'timeout',
 'intercase_n_1__Ca

In [5]:
activity_count = [
 'Callback timeout',
 'Export result',
 'Export to EMS',
 'Match patient data',
 'Receive sample state',
 'Send notification',
 'Wait for plate validation',
 'timeout',
 ]

ii1 = [
 'intercase_n_1__Callback timeout',
 'intercase_n_1__Export result',
 'intercase_n_1__Export to EMS',
 'intercase_n_1__Match patient data',
 'intercase_n_1__Receive sample state',
 'intercase_n_1__Send notification',
 'intercase_n_1__Wait for plate validation',
 'intercase_n_1__timeout'
]

ii3 = [
 'intercase_n_3__Export result_Callback timeout_Send notification',
 'intercase_n_3__Export result_Export to EMS_Callback timeout',
 'intercase_n_3__Export result_Export to EMS_Send notification',
 'intercase_n_3__Export result_Send notification_Callback timeout',
 'intercase_n_3__Export to EMS_Callback timeout_Send notification',
 'intercase_n_3__Export to EMS_Export result_Callback timeout',
 'intercase_n_3__Export to EMS_Export result_Send notification',
 'intercase_n_3__Export to EMS_Send notification_Callback timeout',
 'intercase_n_3__Match patient data',
 'intercase_n_3__Match patient data_Match patient data',
 'intercase_n_3__Match patient data_Match patient data_Match patient data',
 'intercase_n_3__Match patient data_Match patient data_Receive sample state',
 'intercase_n_3__Match patient data_Match patient data_Send notification',
 'intercase_n_3__Match patient data_Receive sample state_Callback timeout',
 'intercase_n_3__Match patient data_Receive sample state_Export result',
 'intercase_n_3__Match patient data_Receive sample state_Export to EMS',
 'intercase_n_3__Match patient data_Receive sample state_Send notification',
 'intercase_n_3__Match patient data_Send notification_Receive sample state',
 'intercase_n_3__Match patient data_Wait for plate validation',
 'intercase_n_3__Match patient data_Wait for plate validation_Receive sample state',
 'intercase_n_3__Match patient data_Wait for plate validation_Send notification',
 'intercase_n_3__Match patient data_Wait for plate validation_timeout',
 'intercase_n_3__Match patient data_timeout',
 'intercase_n_3__Match patient data_timeout_Match patient data',
 'intercase_n_3__Match patient data_timeout_Receive sample state',
 'intercase_n_3__Match patient data_timeout_Send notification',
 'intercase_n_3__Match patient data_timeout_Wait for plate validation',
 'intercase_n_3__Receive sample state_Callback timeout_Send notification',
 'intercase_n_3__Receive sample state_Export result_Export to EMS',
 'intercase_n_3__Receive sample state_Export result_Send notification',
 'intercase_n_3__Receive sample state_Export to EMS_Export result',
 'intercase_n_3__Receive sample state_Export to EMS_Send notification',
 'intercase_n_3__Receive sample state_Send notification_Callback timeout',
 'intercase_n_3__Receive sample state_Send notification_Export result',
 'intercase_n_3__Receive sample state_Send notification_Export to EMS',
 'intercase_n_3__Send notification_Export result_Export to EMS',
 'intercase_n_3__Send notification_Export to EMS_Export result',
 'intercase_n_3__Send notification_Receive sample state_Callback timeout',
 'intercase_n_3__Send notification_Receive sample state_Export result',
 'intercase_n_3__Send notification_Receive sample state_Export to EMS',
 'intercase_n_3__Wait for plate validation',
 'intercase_n_3__Wait for plate validation_Match patient data',
 'intercase_n_3__Wait for plate validation_Match patient data_Receive sample state',
 'intercase_n_3__Wait for plate validation_Match patient data_Send notification',
 'intercase_n_3__Wait for plate validation_Match patient data_timeout',
 'intercase_n_3__Wait for plate validation_Receive sample state',
 'intercase_n_3__Wait for plate validation_Receive sample state_Callback timeout',
 'intercase_n_3__Wait for plate validation_Receive sample state_Export result',
 'intercase_n_3__Wait for plate validation_Receive sample state_Export to EMS',
 'intercase_n_3__Wait for plate validation_Receive sample state_Send notification',
 'intercase_n_3__Wait for plate validation_Send notification_Receive sample state',
 'intercase_n_3__Wait for plate validation_timeout',
 'intercase_n_3__Wait for plate validation_timeout_Match patient data',
 'intercase_n_3__Wait for plate validation_timeout_Receive sample state',
 'intercase_n_3__Wait for plate validation_timeout_Send notification',
 'intercase_n_3__timeout',
 'intercase_n_3__timeout_Match patient data',
 'intercase_n_3__timeout_Match patient data_Match patient data',
 'intercase_n_3__timeout_Match patient data_Receive sample state',
 'intercase_n_3__timeout_Match patient data_Send notification',
 'intercase_n_3__timeout_Match patient data_Wait for plate validation',
 'intercase_n_3__timeout_Receive sample state_Callback timeout',
 'intercase_n_3__timeout_Receive sample state_Export result',
 'intercase_n_3__timeout_Receive sample state_Export to EMS',
 'intercase_n_3__timeout_Receive sample state_Send notification',
 'intercase_n_3__timeout_Send notification_Receive sample state',
 'intercase_n_3__timeout_Wait for plate validation',
 'intercase_n_3__timeout_Wait for plate validation_Match patient data',
 'intercase_n_3__timeout_Wait for plate validation_Receive sample state',
 'intercase_n_3__timeout_Wait for plate validation_Send notification'
]

feature_combinations = {
        'A' : ['concept:name'],
        'AS' : ['concept:name', 'seconds_in_day'],
        'ASAC' : ['concept:name', 'seconds_in_day'] + activity_count,
        'ASD' : ['concept:name', 'seconds_in_day', 'day_of_week'],
        'ASDII1' : ['concept:name', 'seconds_in_day', 'day_of_week'] + ii1,
        'ASDII3' : ['concept:name', 'seconds_in_day', 'day_of_week'] + ii3,
        'ASDACII1' : ['concept:name', 'seconds_in_day', 'day_of_week'] + activity_count + ii1,
        'ASDACII3' : ['concept:name', 'seconds_in_day', 'day_of_week'] + activity_count + ii3
}

In [6]:
quantile_regression_models = {}
for k, v in tqdm(feature_combinations.items()):
    qrm = QuantileRegression(train_data, v)
    qrm.fit()
    quantile_regression_models[k] = qrm

  0%|          | 0/8 [00:00<?, ?it/s]

In [7]:
out_path = './quantile_regression_models.pkl'
with open(out_path, 'wb') as out_file:
    pickle.dump(quantile_regression_models, out_file, protocol=pickle.HIGHEST_PROTOCOL)